# Imports

In [1]:
!pip install datasets lazypredict

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/udev/requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from lazypredict.Supervised import LazyClassifier
from transformers import AutoModelForSequenceClassification, AutoModel
from huggingface_hub import notebook_login
from typing import Tuple
from sklearn.svm import SVC

# Dataset Initialization and Creation

In [24]:
# Load your chosen dataset from HuggingFace
dataset_name = "GleghornLab/DL2_reg"
data = load_dataset(dataset_name)
data

README.md:   0%|          | 0.00/674 [00:00<?, ?B/s]

(…)-00000-of-00001-b378cdf4344ecc7e.parquet:   0%|          | 0.00/2.65M [00:00<?, ?B/s]

(…)-00000-of-00001-68296024fbc161f2.parquet:   0%|          | 0.00/648k [00:00<?, ?B/s]

(…)-00000-of-00001-8feea2b5edb335ed.parquet:   0%|          | 0.00/892k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5473 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/1335 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1729 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 5473
    })
    valid: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 1335
    })
    test: Dataset({
        features: ['seqs', 'labels'],
        num_rows: 1729
    })
})

In [25]:
df_train = pd.DataFrame(data["train"]).drop_duplicates(subset=['seqs'])
df_test = pd.DataFrame(data["test"]).drop_duplicates(subset=['seqs'])
df_valid = pd.DataFrame(data["valid"]).drop_duplicates(subset=['seqs'])
seq_list_train = df_train['seqs'].tolist()
seq_list_test = df_test['seqs'].tolist()
seq_list_valid = df_valid['seqs'].tolist()

# We extract our labels and sequences from our target dataset
# Also note if you simply want to test a preexisting models embeddings that have already been transformed, you can run the first cell of model training and then skip to the hyperparameter tuning section

In [26]:
model_name = "Synthyra/ESMplusplus_small"
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16).cuda()

In [27]:
all_seqs = data['train']['seqs'] + data['valid']['seqs'] + data['test']['seqs']
print(f"Number of sequences: {len(all_seqs)}")
emb_dict = model.embed_dataset(
  sequences=all_seqs, # list of protein strings
  batch_size=8, # embedding batch size
  max_len=20000, # truncate to max_len
  full_embeddings=True, # return residue-wise embeddings
  full_precision=False, # store as float32
  num_workers=0, # data loading num workers
  sql=False, # return dictionary of sequences and embeddings
)

Number of sequences: 8537


Embedding batches:   0%|          | 0/1063 [00:00<?, ?it/s]

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [29]:
class Pooler:
    def __init__(self, hf_dataset, emb_dict):
        self.train_seqs, self.train_labels = hf_dataset['train']['seqs'], hf_dataset['train']['labels']
        self.valid_seqs, self.valid_labels = hf_dataset['valid']['seqs'], hf_dataset['valid']['labels']
        self.test_seqs, self.test_labels = hf_dataset['test']['seqs'], hf_dataset['test']['labels']
        self.emb_dict = emb_dict

        self.pooling_options = {
            'mean': self.mean_pooling,
            'max': self.max_pooling,
            'min': self.min_pooling,
            'norm': self.norm_pooling,
            'prod': self.prod_pooling,
            'median': self.median_pooling,
            'std': self.std_pooling,
            'var': self.var_pooling,
            'cls': self.cls_pooling,
            'eos': self.eos_pooling
        }

    def mean_pooling(self, emb): # (L, d) -> (d)
        return emb.mean(dim=0)

    def max_pooling(self, emb): # (L, d) -> (d)
        return emb.max(dim=0).values

    def min_pooling(self, emb): # (L, d) -> (d)
        return emb.min(dim=0).values

    def norm_pooling(self, emb): # (L, d) -> (d)
        return emb.norm(dim=0, p=2)

    def prod_pooling(self, emb): # (L, d) -> (d)
        length = emb.shape[0]
        return emb.prod(dim=0) / length

    def median_pooling(self, emb): # (L, d) -> (d)
        return emb.median(dim=0).values

    def std_pooling(self, emb): # (L, d) -> (d)
        return emb.std(dim=0)

    def var_pooling(self, emb): # (L, d) -> (d)
        return emb.var(dim=0)

    def cls_pooling(self, emb): # (L, d) -> (d)
        return emb[0, :]

    def eos_pooling(self, emb): # (L, d) -> (d)
        return emb[-1, :]

    def get_pooled_embeddings(self, pooling_types: Tuple[str, str]): # [mean, max]
        """
        pooler = Pooler(hf_dataset, emb_dict)
        pooling_types = ['mean', 'max']
        X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)
        # X (n_samples, n_features)
        # y (n_samples)
        """
        for pool_type in pooling_types:
            if pool_type not in self.pooling_options:
                raise ValueError(f"Invalid pooling type: {pool_type}")

        train_embeddings, valid_embeddings, test_embeddings = [], [], []
        train_labels = np.array(self.train_labels)
        valid_labels = np.array(self.valid_labels)
        test_labels = np.array(self.test_labels)

        # concatenate all pooling for each sequence
        for seq in self.train_seqs:
            final_emb = []
            emb = self.emb_dict[seq] # (L, d)
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb) # (d)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten() # (d * len(pooling_type))
            train_embeddings.append(final_emb)

        train_embeddings = torch.stack(train_embeddings).numpy().astype(np.float32)

        for seq in self.valid_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            valid_embeddings.append(final_emb)

        valid_embeddings = torch.stack(valid_embeddings).numpy().astype(np.float32)

        for seq in self.test_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            test_embeddings.append(final_emb)

        test_embeddings = torch.stack(test_embeddings).numpy().astype(np.float32)

        return train_embeddings, train_labels, valid_embeddings, valid_labels, test_embeddings, test_labels


In [30]:
pooling_types = ['mean']
pooler = Pooler(data, emb_dict)
X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)

# Model Training and Evaluation

In [31]:
def mcc_metric(y_true, y_pred):
    return matthews_corrcoef(y_true, y_pred)

# Set a display name for the function
mcc_metric.__name__ = "mcc_metric"

# Create a sklearn scorer for MCC
mcc_scorer = make_scorer(mcc_metric, greater_is_better=True)

In [32]:
reg = LazyClassifier(verbose=0, ignore_warnings=False, custom_metric=mcc_metric)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)
print(f'Model Name: {model_name[10:]}')
print(models)
simplified_table = models[['mcc_metric']].copy()
simplified_table['Model'] = simplified_table.index
simplified_table = simplified_table[['Model', 'mcc_metric']]
csv_filename = f"mcc_metrics_{dataset_name[12:]}.csv"
simplified_table.to_csv(csv_filename, index=False)
print(f"File saved as {csv_filename}")


 16%|█▌        | 5/32 [02:59<13:29, 29.99s/it]

CategoricalNB model failed to execute
Negative values in data passed to CategoricalNB (input X).


 34%|███▍      | 11/32 [03:12<01:31,  4.37s/it]

FixedThresholdClassifier model failed to execute
FixedThresholdClassifier.__init__() missing 1 required positional argument: 'estimator'


 88%|████████▊ | 28/32 [04:35<00:14,  3.64s/it]

SelfTrainingClassifier model failed to execute
You must pass an estimator to SelfTrainingClassifier. Use `estimator`.
StackingClassifier model failed to execute
StackingClassifier.__init__() missing 1 required positional argument: 'estimators'
TunedThresholdClassifierCV model failed to execute
TunedThresholdClassifierCV.__init__() missing 1 required positional argument: 'estimator'


 97%|█████████▋| 31/32 [04:42<00:02,  2.80s/it]

XGBClassifier model failed to execute
'super' object has no attribute '__sklearn_tags__'
[LightGBM] [Info] Number of positive: 3227, number of negative: 2246
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022303 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 244800
[LightGBM] [Info] Number of data points in the train set: 5473, number of used features: 960
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.589622 -> initscore=0.362402
[LightGBM] [Info] Start training from score 0.362402


100%|██████████| 32/32 [04:45<00:00,  8.93s/it]

Model Name: SMplusplus_small
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
SVC                                0.92               0.91     0.91      0.92   
LogisticRegression                 0.88               0.88     0.88      0.88   
RidgeClassifierCV                  0.88               0.88     0.88      0.88   
SGDClassifier                      0.88               0.88     0.88      0.88   
RidgeClassifier                    0.88               0.87     0.87      0.88   
LinearDiscriminantAnalysis         0.88               0.87     0.87      0.88   
LGBMClassifier                     0.88               0.86     0.86      0.88   
Perceptron                         0.86               0.86     0.86      0.86   
CalibratedClassifierCV             0.87               0.86     0.86      0.87   
PassiveAggressiveClassifier        0.86               0.86     0.86      0.86   

In [ ]:
pearson_multiple(embeddings)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (1511,) + inhomogeneous part.

# Hyperparameter Tuning

In [ ]:
split = "username/split_name" # change split name here for testing/tuning
embeddings = load_dataset(f"{split}") # If you already have a dataset with transformed embeddings you can load it here and start tuning hyperparameters and adjusting
X_train = np.array(embeddings['train']['vectors'], dtype='object')
X_valid = np.array(embeddings['validation']['vectors'])
X_test = np.array(embeddings['test']['vectors'], dtype='object')
y_train = np.array(df_train['labels'])
y_valid = np.array(df_valid['labels'])
y_test = np.array(df_test['labels'])

In [38]:
model = SVC() # You can use any other model provided but it must be imported

param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization parameter
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],  # Kernel type
    'degree': [2, 3, 4, 5],  # Degree of the polynomial kernel function (only for 'poly')
    'gamma': ['scale', 'auto'],  # Kernel coefficient
    'coef0': [0.1, 0.5, 1, 2],  # Independent term in kernel function (for 'poly' and 'sigmoid')
    'shrinking': [True, False],  # Whether to use the shrinking heuristic
    'probability': [True],  # Whether to enable probability estimates (adds computational cost)
    'class_weight': [None, 'balanced'],  # Adjust weights inversely proportional to class frequencies
    'decision_function_shape': ['ovo', 'ovr'],  # One-vs-One or One-vs-Rest
    'tol': [1e-3, 1e-4, 1e-5],  # Tolerance for stopping criterion
    'max_iter': [-1, 1000, 5000],  # Limit on iterations (-1 for no limit)
}
# Provide a parameter grid to test a wide range of parameters for your chosen model to optimize

grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=5, scoring=mcc_scorer, verbose=4, n_iter = 20) # Creates a grid search instance, note that the scoring method can also be adjusted here

In [39]:
grid_search.fit(X_train, y_train) # Fits the parameter grid on a random set of hyperparameters provided in the parameter grid

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END C=10, class_weight=balanced, coef0=0.5, decision_function_shape=ovr, degree=5, gamma=auto, kernel=linear, max_iter=5000, probability=True, shrinking=False, tol=1e-05;, score=0.809 total time=  17.4s
[CV 2/5] END C=10, class_weight=balanced, coef0=0.5, decision_function_shape=ovr, degree=5, gamma=auto, kernel=linear, max_iter=5000, probability=True, shrinking=False, tol=1e-05;, score=0.805 total time=  17.0s
[CV 3/5] END C=10, class_weight=balanced, coef0=0.5, decision_function_shape=ovr, degree=5, gamma=auto, kernel=linear, max_iter=5000, probability=True, shrinking=False, tol=1e-05;, score=0.653 total time=  17.1s
[CV 4/5] END C=10, class_weight=balanced, coef0=0.5, decision_function_shape=ovr, degree=5, gamma=auto, kernel=linear, max_iter=5000, probability=True, shrinking=False, tol=1e-05;, score=0.728 total time=  17.0s
[CV 5/5] END C=10, class_weight=balanced, coef0=0.5, decision_function_shape=ovr, degree=5

KeyboardInterrupt: 

In [ ]:
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_
best_score = grid_search.best_score_

print(best_params)
print(best_estimator) # Shows the best hyperparameters for the model